In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf

In [ ]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN"]

data = yf.download(tickers, start="2020-01-01", end="2024-01-01")["Close"]
data = data.dropna()

In [ ]:
returns = data.pct_change().dropna()

mean_returns = returns.mean() * 252
cov_matrix = returns.cov().values * 252

In [ ]:
lambda_ = 0.1  # shrinkage intensity

I = np.eye(cov_matrix.shape[0])
cov_shrunk = (1 - lambda_) * cov_matrix + lambda_ * I

In [ ]:
def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns)

def portfolio_volatility(weights, cov):
    return np.sqrt(weights.T @ cov @ weights)

def negative_sharpe(weights, mean_returns, cov, rf=0):
    ret = portfolio_return(weights, mean_returns)
    vol = portfolio_volatility(weights, cov)
    return -(ret - rf) / vol

In [ ]:
n_assets = len(mean_returns)

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = tuple((0, 1) for _ in range(n_assets))

x0 = np.ones(n_assets) / n_assets

In [ ]:
result = minimize(
    negative_sharpe,
    x0,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

weights_original = result.x

In [ ]:
result_shrunk = minimize(
    negative_sharpe,
    x0,
    args=(mean_returns, cov_shrunk),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

weights_shrunk = result_shrunk.x

In [ ]:
comparison = pd.DataFrame({
    "Original": weights_original,
    "Shrunk": weights_shrunk
}, index=returns.columns)

print(comparison)

In [ ]:
ret_orig = portfolio_return(weights_original, mean_returns)
vol_orig = portfolio_volatility(weights_original, cov_matrix)

ret_shr = portfolio_return(weights_shrunk, mean_returns)
vol_shr = portfolio_volatility(weights_shrunk, cov_shrunk)

print("\nOriginal Portfolio:")
print("Return:", ret_orig, "Volatility:", vol_orig)

print("\nShrunk Portfolio:")
print("Return:", ret_shr, "Volatility:", vol_shr)

In [ ]:
n_portfolios = 5000

results = np.zeros((3, n_portfolios))

for i in range(n_portfolios):
    weights = np.random.random(n_assets)
    weights /= np.sum(weights)

    ret = portfolio_return(weights, mean_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    sharpe = ret / vol

    results[0, i] = vol
    results[1, i] = ret
    results[2, i] = sharpe

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(results[0], results[1], c=results[2], cmap='viridis')

plt.colorbar(label='Sharpe Ratio')

plt.scatter(vol_orig, ret_orig, label="Original", s=100)
plt.scatter(vol_shr, ret_shr, label="Shrunk", s=100)

plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title("Efficient Frontier with Shrinkage Comparison")
plt.legend()

plt.show()

## Key Insights

- Mean-variance optimization is highly sensitive to covariance estimates.
- Covariance shrinkage reduces estimation noise and leads to more stable portfolio weights.
- The shrunk portfolio is typically more robust and less prone to extreme allocations.

This highlights a key limitation of classical Markowitz optimization and the importance of regularization.